# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    last_err = None
    for attempt in range(tries):
      try:
            r = requests.get(BASE, params=params, timeout=10)
      except (requests.ConnectionError, requests.Timeout) as e:
            last_err = e
      else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                raise requests.HTTPError(
                    f'{r.status_code} from Open-Meteo: {r.text[:200]}', response=r)
            last_err = requests.HTTPError(f'{r.status_code} (retryable)', response=r)
      if attempt < tries - 1:
          time.sleep(2 ** attempt)
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed
    raise RuntimeError(f'get_weather failed after {tries} attempts') from last_err

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about

used_fallback = False
try:
    raw = get_weather('2026-09-05', '2026-10-03')
    wx = raw[raw['time'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
    pulled_at = pd.Timestamp.now(tz='America/New_York')
except (requests.RequestException, RuntimeError) as e:
    wx = None
    print(f'Weather fetch failed, will need fallback: {e}')


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
if wx is None:
    used_fallback = True
    wx = FALLBACK.copy()
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wet = wx.loc[wx['precip_mm'].idxmax()]
dry = wx.loc[wx['precip_mm'].idxmin()]
diff = wet['precip_mm'] - dry['precip_mm']

if diff == 0:
    print(f"Both weekends had the same rainfall: {wet['precip_mm']:.1f} mm "
          f"({wet['source']} data).")
else:
    print(f"{wet['date']:%A, %B %-d, %Y} was the wetter day, with "
          f"{wet['precip_mm']:.1f} mm of rain versus {dry['precip_mm']:.1f} mm on "
          f"{dry['date']:%A, %B %-d}: {diff:.1f} mm more ({wet['source']} data).")

Saturday, October 3, 2026 was the wetter day, with 14.7 mm of rain versus 2.5 mm on Saturday, September 5: 12.2 mm more (open-meteo data).


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
print(f"Endpoint:    {BASE}")
print(f"Coordinates: 38.03, -78.51 (Charlottesville, VA)")
print(f"Date range:  2026-09-05 to 2026-10-03 (kept 2026-09-05 and 2026-10-03)")
if used_fallback:
    print("Pulled:      not pulled; the fetch failed")
    print("Data:        FALLBACK SAMPLE, not real observations")
else:
    print(f"Pulled:      {pulled_at:%Y-%m-%d %H:%M %Z}")
    print("Data:        real, from Open-Meteo historical archive")

Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51 (Charlottesville, VA)
Date range:  2026-09-05 to 2026-10-03 (kept 2026-09-05 and 2026-10-03)
Pulled:      2026-10-11 17:43 EDT
Data:        real, from Open-Meteo historical archive


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

The merge will run without any errors, but every weather value will come back as NaN. Then, the chart won't show anything, which means in the context of the chart that rain won't affect revenue. Nobody will catch this because there's no error, the row count looks correct, and the conclusion is pretty believable.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
DATES = ['2026-09-05', '2026-10-03']

orl_raw = get_weather(DATES[0], DATES[1], lat=28.54, lon=-81.38)
orl = orl_raw[orl_raw['time'].isin(DATES)].rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
}).copy()
orl['date'] = pd.to_datetime(orl['date'])
orl['source'] = 'open-meteo'
orl['city'] = 'Orlando'

cvl = wx.assign(city='Charlottesville')

both = pd.concat([cvl, orl], ignore_index=True)[
    ['city', 'date', 'temp_max', 'precip_mm', 'source']
]

assert len(both) == 4, f'expected 4 rows, got {len(both)}'
assert both.groupby('city').size().eq(2).all()
assert pd.api.types.is_datetime64_any_dtype(both['date'])
assert (both['precip_mm'] >= 0).all()
both

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
